
<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Классификация изображений</b></h3>


# Домашнее задание. Классификация изображений

Сегодня вам предстоить помочь телекомпании FOX в обработке их контента. Как вы знаете, сериал "Симпсоны" идет на телеэкранах более 25 лет, и за это время скопилось очень много видеоматериала. Персоонажи менялись вместе с изменяющимися графическими технологиями, и Гомер Симпсон-2018 не очень похож на Гомера Симпсона-1989. В этом задании вам необходимо классифицировать персонажей, проживающих в Спрингфилде. Думаю, нет смысла представлять каждого из них в отдельности.



В нашем тесте будет 991 картинка, для которых вам будет необходимо предсказать класс.

## Шаг 1. Установка зависимостей

#### Установим необходимые библиотеки и проверим доступность CUDA

In [63]:
# we will verify that GPU is enabled for this notebook
# following should print: CUDA is available!  Training on GPU ...
#
# if it prints otherwise, then you need to enable GPU:
# from Menu > Runtime > Change Runtime Type > Hardware Accelerator > GPU
import torch
import numpy as np

train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')

CUDA is available!  Training on GPU ...


In [64]:
!nvidia-smi

Sat Apr 18 13:00:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   41C    P8             16W /  300W |    4123MiB /  16303MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [65]:
import pickle
import numpy as np
from skimage import io

from tqdm import tqdm, tqdm_notebook
from PIL import Image
from pathlib import Path

from torchvision import transforms
from torchvision.transforms import v2

import torchsummary

from multiprocessing.pool import ThreadPool
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from matplotlib import colors, pyplot as plt
%matplotlib inline
from sklearn.metrics import f1_score

import torch.optim as optim
from torchvision import models
import random, numpy as np, os

# в sklearn не все гладко, чтобы в colab удобно выводить картинки
# мы будем игнорировать warnings
import warnings
warnings.filterwarnings(action='ignore', category=DeprecationWarning)


 #### Определим константы, которые будем использовать в по ходу ноутбука

In [66]:
# разные режимы датасета
DATA_MODES = ['train', 'val', 'test']

# работаем на видеокарте
DEVICE = torch.device("cuda")
# PATH = 'content/journey-to-springfield1'
PATH = 'content/'
#определим директории с тренировочными и тестовыми файлами
TRAIN_DIR = Path(f'{PATH}/train') #Path('./data/train/')
TEST_DIR = Path(f'{PATH}/testset') #Path('./data/testset')

# параметры нормировки изображений по трем каналам перед подачей в модель
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

# все изображения будут масштабированы к размеру 224x224 px
RESCALE_SIZE = [224, 224]

## Шаг 2. Загрузка и обработка данных

#### Скачаем изображения по ссылке

Посмотрите на структуру файлов в папках train и testset.

В train лежат данные, которые мы будем использовать для обучения модели. Изображения персонажей разложены по папкам, которые названы по именам персонажей. Названия папок мы в дальнейшем будет использовать в качестве текстовых меток классов.

В testset находятся изображения, для которых вам надо будет сделать прогноз наиболее вероятного класса.


Для обращения к файлам сформируем списки имен файлов обучающей+валидационнной и тестовой выборок. Это полные имена, включающие путь к файлам.


In [67]:
train_val_files = sorted(list(TRAIN_DIR.rglob('*.jpg')))
test_files = sorted(list(TEST_DIR.rglob('*.jpg')))

In [68]:
print(f"Train files: {len(train_val_files)}")
print(f"Test files: {len(test_files)}")

Train files: 20933
Test files: 991


Кодировать имена персонажей в числовые метки класса и обратно будем при помощи `LabelEncoder`.

Для train выборки сформируем список текстовых меток всех изображений - имя родительской директории, которая одновременно является и именем персонажа. Зададим числовые метки классов нашего энкодера при помощи метода `fit`.

Далее будем применять метод `transform` для преобразования текстовых меток в числовые, и метод `inverse_transform` для преобразования числовых меток в текстовые.


In [69]:
label_encoder = LabelEncoder()

train_val_labels = [path.parent.name for path in train_val_files]

label_encoder.fit(train_val_labels)
num_classes = len(label_encoder.classes_)

Разделим train выборку на обучающую и валидационнную части. Для того, чтобы персонажи были пропорционально представлены в обучающей и валидационнной подвыборках, применим стратификацию по меткам класса.

In [70]:
from sklearn.model_selection import train_test_split

train_files, val_files = train_test_split(train_val_files, test_size=0.25, \
                                          stratify=train_val_labels)

#### Создадим Datasets и Dataloaders

Важно разобраться, что делает метод self.transform_images_to_tensors().

`Compose` объединяет последовательность следующих преобразований:
- `PILToTensor` конвертирует  `PIL Image` в тензор с параметрами в диапазоне $[0, 255]$ (как все пиксели в исходном изображении)
- `ToDtype` преобразует тензор в `FloatTensor` размера ($C \times H \times W$) со значениями пикселей в диапазоне $[0,1]$
- затем `Normalize` производится масштабирование:
$\text{input} = \frac{\text{input} - \text{mean}}{\text{std}} $, <br>      где константы mean и std - средние и дисперсии по каналам в датасете ImageNet
- наконец, `Resize` преобразует картинки к размеру $224 \times 224$ (в описании датасета указано, что картинки разного размера, так как брались напрямую с видео, поэтому следует привести их к одному размеру).

Сейчас аугментация не изображений не производится, поэтому для обучающих и валидационных/тестовых изображений производится одинаковая трансформация. В дальнейшем, если вы захотите добавить аугментацию, вы можете сделать это, например, модифицировав метод `transform_images_to_tensors`. Подробнее про трансформацию изображений вы можете почитать в документации: https://docs.pytorch.org/vision/main/transforms.html


In [71]:
class SimpsonsDataset(Dataset):
    def __init__(self, files, label_encoder, mode, augment_type='basic'):
        super().__init__()
        # список файлов для загрузки
        self.files = sorted(files)
        # режим работы
        self.mode = mode
        if self.mode not in DATA_MODES:
            print(f"{self.mode} is not correct; correct modes: {DATA_MODES}")
            raise NameError

        self.label_encoder = label_encoder
        self.len_ = len(self.files)

        self.augment_type = augment_type  # <- флажок для аугументации

    def __len__(self):
        return self.len_ # сейчас self.__len__() возвращает количество картинок, подаваемых на вход.
        # Если вы решите перевзвесить размеры категорий внутри класса -
        # не забудьте изменить вывод self.__len__()

    def __getitem__(self, index):
        x = self.load_image(self.files[index])
        x = self.transform_images_to_tensors(x)

        if self.mode == 'test':
            return x
        else:
            path = self.files[index]
            y = self.label_encoder.transform([path.parent.name,]).item()
            return x, y

    # принимает путь к файлу изображения и возвращает само изображение
    def load_image(self, file):
        image = Image.open(file)
        image.load()
        return image

    # преобразует изображение в тензор
    def transform_images_to_tensors(self, image):
      if self.mode == 'train':
        if self.augment_type == 'basic':
            transform = v2.Compose([
                v2.Resize(RESCALE_SIZE),
                v2.PILToTensor(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
                
            ])
        elif self.augment_type == 'custom':
            transform = v2.Compose([
            v2.Resize(RESCALE_SIZE),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomRotation(degrees=15),
            v2.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.1
            ),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
            ])
        
      else:
        transform = v2.Compose([
            v2.Resize(RESCALE_SIZE),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
            
          ])

      tensor_transformed = transform(image)
      return(tensor_transformed)


In [72]:
train_dataset = SimpsonsDataset(train_files, label_encoder = label_encoder, mode='train', augment_type='custom')
val_dataset = SimpsonsDataset(val_files, label_encoder, mode='val')

In [73]:
batch_size = 128

In [74]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
loaders = {'train':train_loader, 'val': val_loader}

Напишите функции:
- для обучения модели на одной эпохе
- для валидации модели на одной эпохе
- для реализации полного цикла обучения

За основу можно взять функции, которые вы написали в предыдущем домашнем задании.

In [75]:
from tqdm import tqdm
def train_one_epoch(model, dataloader, loss_func, optimizer, device):
    """
    Args:
        model: модель PyTorch
        dataloader: DataLoader с обучающей выборкой
        loss_func: функция потерь
        optimizer: оптимизатор
        device: 'cpu', 'cuda' или 'mps'
    
    Returns:
        epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels
    """
    model.train()
    all_preds = []
    all_labels = []
    running_loss = 0.0

    for images, labels in tqdm(dataloader, desc='Training', leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    epoch_loss = running_loss / len(all_labels)
    epoch_acc = (all_preds == all_labels).sum().item() / len(all_labels)
    epoch_f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average='micro')

    return epoch_loss, epoch_acc, epoch_f1



In [76]:
def val_one_epoch(model, dataloader, loss_func, device):
    """
    Args:
        model: модель PyTorch
        dataloader: DataLoader с валидационной выборкой
        loss_func: функция потерь
        device: 'cpu', 'cuda' или 'mps'
    
    Returns:
        epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels
    """
    model.eval()
    all_preds = []
    all_labels = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_func(outputs, labels)
            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    epoch_loss = running_loss / len(all_labels)
    epoch_acc = (all_preds == all_labels).sum().item() / len(all_labels)
    epoch_f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average='micro')

    return epoch_loss, epoch_acc, epoch_f1

In [77]:
def train_model(model, train_loader, val_loader, loss_func, optimizer, num_epochs=10, device=None):
    """
    Args:
        model: PyTorch модель
        train_loader: DataLoader для обучения
        val_loader: DataLoader для валидации
        loss_func: функция потерь
        optimizer: оптимизатор
        num_epochs: количество эпох
        device: 'cpu', 'cuda' или 'mps'. Если None, выбирается автоматически.
    
    Returns:
        model: обученная модель
        history: словарь с историей метрик
    """

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else 
                              "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Training on device: {device}")
    
    model.to(device)

    best_f1 = 0.0
    history = {"train_loss": [], "train_acc": [], "train_f1": [],
               "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # Тренировка 
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, loss_func, optimizer, device)
        
        #Валидация
        val_loss, val_acc, val_f1 = val_one_epoch(model, val_loader, loss_func, device)

        # Вывод
        print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

        #Сохранение истории
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        #Сохраняем лучшую модель по F1
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_model.pth")

    print(f"\nBest Validation F1: {best_f1:.4f}")
    return model, history


In [78]:
# вобще можно было выводить только accuracy или f1-score(micro) - т.к. это одно и тоже, но пусть будет) 

#### Модель

In [79]:
class SimpleCnn(nn.Module):
    """
    Очень простая сверточная нейронная сеть для классификации изображений.

    Эта сеть состоит из пяти сверточных слоев, каждый из которых
    включает в себя операцию свертки, функцию активации ReLU и операцию
    пулинга (max-pooling). На выходе используется полносвязный слой
    для классификации на заданное количество классов.

    Параметры:
    ----------
    n_classes : int
        Количество классов для классификации.

    Примечание:
    ----------
    Входные изображения должны иметь размерность (3, H, W), где
    3 - слои rgb для цветной картинки, а H и W - высота и ширина изображения,
    соответственно. Размер выходного тензора будет равен (n_classes).

    Методы:
    -------
    forward(x):
        Пропускает входные данные через сеть и возвращает логиты для
        каждого класса.
    """

    def __init__(self, n_classes):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=96, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.out = nn.Linear(96 * 5 * 5, n_classes)


    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)

        x = x.view(x.size(0), -1)
        logits = self.out(x)
        
        return logits

*Описание слоев*:

1. размерность входа: $3\times 224 \times 224$
2. размерность после 1-го слоя (Conv2d + ReLU + MaxPool2d):  $8 \times 111 \times 111$
3. после 2-го слоя: $16 \times 54 \times 54$
4. после 3-го слоя: $32 \times 26 \times 26$
5. после 4-го слоя: $64 \times 12 \times 12$
6. после 5-го слоя: $96 \times 5 \times 5$
7. после полносвязного слоя (выход модели): количество классов

In [80]:
model_simple_cnn = SimpleCnn(n_classes = len(np.unique(train_val_labels)))
model_simple_cnn.to(DEVICE)
torchsummary.summary(model_simple_cnn, (3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 8, 222, 222]             224
              ReLU-2          [-1, 8, 222, 222]               0
         MaxPool2d-3          [-1, 8, 111, 111]               0
            Conv2d-4         [-1, 16, 109, 109]           1,168
              ReLU-5         [-1, 16, 109, 109]               0
         MaxPool2d-6           [-1, 16, 54, 54]               0
            Conv2d-7           [-1, 32, 52, 52]           4,640
              ReLU-8           [-1, 32, 52, 52]               0
         MaxPool2d-9           [-1, 32, 26, 26]               0
           Conv2d-10           [-1, 64, 24, 24]          18,496
             ReLU-11           [-1, 64, 24, 24]               0
        MaxPool2d-12           [-1, 64, 12, 12]               0
           Conv2d-13           [-1, 96, 10, 10]          55,392
             ReLU-14           [-1, 96,

In [81]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_simple_cnn.parameters(), lr = 1e-3)

Запустите обучение сети

In [82]:
# YOUR CODE

num_epochs = 10

model, history = train_model(
    model=model_simple_cnn,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs,
    device=DEVICE
)

# Train Loss: 2.9133 | Acc: 0.1885 | F1: 0.1885
# Val   Loss: 2.4394 | Acc: 0.3233 | F1: 0.3233

# Epoch 2/10
                                                           
# Train Loss: 2.1178 | Acc: 0.4219 | F1: 0.4219
# Val   Loss: 1.7923 | Acc: 0.5086 | F1: 0.5086

# Epoch 3/10
                                                           
# Train Loss: 1.6010 | Acc: 0.5601 | F1: 0.5601
# Val   Loss: 1.4130 | Acc: 0.6143 | F1: 0.6143

# Epoch 4/10
                                                           
# Train Loss: 1.2538 | Acc: 0.6548 | F1: 0.6548
# Val   Loss: 1.1341 | Acc: 0.6870 | F1: 0.6870

# Epoch 5/10
                                                           
# Train Loss: 1.0230 | Acc: 0.7208 | F1: 0.7208
# Val   Loss: 0.9721 | Acc: 0.7371 | F1: 0.7371

# Epoch 6/10
                                                           
# Train Loss: 0.8538 | Acc: 0.7654 | F1: 0.7654
# Val   Loss: 0.8901 | Acc: 0.7621 | F1: 0.7621

# Epoch 7/10
                                                           
# Train Loss: 0.7350 | Acc: 0.7960 | F1: 0.7960
# Val   Loss: 0.7971 | Acc: 0.7833 | F1: 0.7833

# Epoch 8/10
                                                           
# Train Loss: 0.6427 | Acc: 0.8192 | F1: 0.8192
# Val   Loss: 0.7683 | Acc: 0.7893 | F1: 0.7893

# Epoch 9/10
                                                           
# Train Loss: 0.5920 | Acc: 0.8355 | F1: 0.8355
# Val   Loss: 0.7698 | Acc: 0.7933 | F1: 0.7933

# Epoch 10/10
#                                                            Train Loss: 0.5191 | Acc: 0.8535 | F1: 0.8535
# Val   Loss: 0.6998 | Acc: 0.8158 | F1: 0.8158

# Best Validation F1: 0.8158

# 13m

Training on device: cuda

Epoch 1/10


Train Loss: 2.9133 | Acc: 0.1885 | F1: 0.1885
Val   Loss: 2.4394 | Acc: 0.3233 | F1: 0.3233

Epoch 2/10


Train Loss: 2.1178 | Acc: 0.4219 | F1: 0.4219
Val   Loss: 1.7923 | Acc: 0.5086 | F1: 0.5086

Epoch 3/10


Train Loss: 1.6010 | Acc: 0.5601 | F1: 0.5601
Val   Loss: 1.4130 | Acc: 0.6143 | F1: 0.6143

Epoch 4/10


Train Loss: 1.2538 | Acc: 0.6548 | F1: 0.6548
Val   Loss: 1.1341 | Acc: 0.6870 | F1: 0.6870

Epoch 5/10


...

Train Loss: 0.8538 | Acc: 0.7654 | F1: 0.7654
Val   Loss: 0.8901 | Acc: 0.7621 | F1: 0.7621

Epoch 7/10


Train Loss: 0.7350 | Acc: 0.7960 | F1: 0.7960
Val   Loss: 0.7971 | Acc: 0.7833 | F1: 0.7833

Epoch 8/10


Train Loss: 0.6427 | Acc: 0.8192 | F1: 0.8192
Val   Loss: 0.7683 | Acc: 0.7893 | F1: 0.7893

Epoch 9/10


Train Loss: 0.5920 | Acc: 0.8355 | F1: 0.8355
Val   Loss: 0.7698 | Acc: 0.7933 | F1: 0.7933

Epoch 10/10


Train Loss: 0.5191 | Acc: 0.8535 | F1: 0.8535
Val   Loss: 0.6998 | Acc: 0.8158 | F1: 0.8158

Best Validation F1: 0.8158


## Шаг 6. Submit на Kaggle

Создадим loader для тестовых данных

In [83]:
test_dataset = SimpsonsDataset(test_files, label_encoder = label_encoder, mode="test")
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=64)

Воспользуемся функцией predict, которая возвращает предсказанные числовые метки для всех объектов в лоадере.

In [84]:
def predict(model, loader):
    model.eval()
    all_predictions = torch.tensor([]).to(DEVICE).int()
    print("Test mode...")
    for inputs in tqdm_notebook(loader):
        inputs = inputs.to(DEVICE)

        with torch.no_grad():
            outputs = model(inputs)

            predictions = outputs.argmax(-1).int()
            all_predictions = torch.cat((all_predictions, predictions), 0)
    return all_predictions.cpu()

Получем предсказание меток классов для тестовых данных:

In [85]:
predicted_numeric_labels = predict(model_simple_cnn, test_loader)

Test mode...


  0%|          | 0/16 [00:00<?, ?it/s]

и преобразуем их в текстовые метки:

In [86]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)

Загрузим пример файла для загрузки на Kaggle (проверьте путь, по которому у вас лежит файл sample_submission.csv и при необходимости скорректируйте путь в коде ниже):

In [87]:
import pandas as pd
sample_submission = pd.read_csv("content/sample_submission.csv")
sample_submission.head(10)

,Id,Expected
0,img0.jpg,bart_simpson
1,img1.jpg,bart_simpson
2,img2.jpg,bart_simpson
3,img3.jpg,bart_simpson
4,img4.jpg,bart_simpson
5,img5.jpg,bart_simpson
6,img6.jpg,bart_simpson
7,img7.jpg,bart_simpson
8,img8.jpg,bart_simpson
9,img9.jpg,bart_simpson


In [88]:
my_submission = pd.DataFrame({'Id': [path.name for path in test_files], 'Expected': predicted_text_labels})
my_submission.head(10)

,Id,Expected
0,img0.jpg,nelson_muntz
1,img1.jpg,bart_simpson
2,img10.jpg,ned_flanders
3,img100.jpg,chief_wiggum
4,img101.jpg,apu_nahasapeemapetilon
5,img102.jpg,kent_brockman
6,img103.jpg,edna_krabappel
7,img104.jpg,chief_wiggum
8,img105.jpg,lisa_simpson
9,img106.jpg,kent_brockman


In [89]:
my_submission.to_csv('content/mixup50.csv', index=False)

## Собственные эксперементы:

### Работа с даннымм 

У нас явно присутсвует дизбаланс в обучающейся выборке, посмотрим какой и потом попробуем его исправить

In [90]:
from collections import Counter

# достаём имена папок 
class_names = [path.parent.name for path in train_dataset.files]

# считаем количество изображений каждого класса
class_counts = Counter(class_names)

# сортируем для наглядности
for cls, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{cls}: {count}")


homer_simpson: 1684
ned_flanders: 1090
moe_szyslak: 1089
lisa_simpson: 1015
bart_simpson: 1006
marge_simpson: 968
krusty_the_clown: 904
charles_montgomery_burns: 895
principal_skinner: 895
milhouse_van_houten: 809
chief_wiggum: 739
abraham_grampa_simpson: 685
sideshow_bob: 658
apu_nahasapeemapetilon: 467
kent_brockman: 373
comic_book_guy: 352
edna_krabappel: 343
nelson_muntz: 269
lenny_leonard: 233
mayor_quimby: 185
waylon_smithers: 136
maggie_simpson: 96
groundskeeper_willie: 91
barney_gumble: 80
selma_bouvier: 77
carl_carlson: 74
ralph_wiggum: 67
patty_bouvier: 54
martin_prince: 53
professor_john_frink: 49
snake_jailbird: 41
cletus_spuckler: 35
rainier_wolfcastle: 34
agnes_skinner: 32
sideshow_mel: 30
otto_mann: 24
fat_tony: 20
gil: 20
miss_hoover: 13
disco_stu: 6
troy_mcclure: 6
lionel_hutz: 2


In [91]:

# Устройство 
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Device:", device)


num_classes = len(label_encoder.classes_)

# ResNet50 без загрузки предобученных весов 
model = models.resnet50(weights=None)  
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

#Оптимизатор
lr_head = 1e-3
lr_backbone = 1e-4
weight_decay = 1e-4

backbone_params = []
head_params = []
for name, p in model.named_parameters():
    if name.startswith("fc."):
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": lr_backbone},
    {"params": head_params, "lr": lr_head}
], weight_decay=weight_decay)


criterion = nn.CrossEntropyLoss()

num_epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Обучение
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())



model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs
)

# Epoch 1/10
                                                           
# Train Loss: 3.2094 | Acc: 0.1162 | F1: 0.1162
# Val   Loss: 2.9013 | Acc: 0.1997 | F1: 0.1997

# Epoch 2/10
                                                           
# Train Loss: 2.5426 | Acc: 0.2999 | F1: 0.2999
# Val   Loss: 2.2737 | Acc: 0.3802 | F1: 0.3802

# Epoch 3/10
                                                           
# Train Loss: 1.7995 | Acc: 0.5135 | F1: 0.5135
# Val   Loss: 1.7035 | Acc: 0.5573 | F1: 0.5573

# Epoch 4/10
                                                           
# Train Loss: 1.2753 | Acc: 0.6503 | F1: 0.6503
# Val   Loss: 1.3223 | Acc: 0.6561 | F1: 0.6561

# Epoch 5/10
                                                           
# Train Loss: 0.9828 | Acc: 0.7278 | F1: 0.7278
# Val   Loss: 0.9128 | Acc: 0.7394 | F1: 0.7394

# Epoch 6/10
                                                           
# Train Loss: 0.7785 | Acc: 0.7822 | F1: 0.7822
# Val   Loss: 0.7843 | Acc: 0.7883 | F1: 0.7883

# Epoch 7/10
                                                           
# Train Loss: 0.6682 | Acc: 0.8120 | F1: 0.8120
# Val   Loss: 0.6884 | Acc: 0.8128 | F1: 0.8128

# Epoch 8/10
                                                           
# Train Loss: 0.5562 | Acc: 0.8413 | F1: 0.8413
# Val   Loss: 0.6492 | Acc: 0.8238 | F1: 0.8238

# Epoch 9/10
                                                           
# Train Loss: 0.4902 | Acc: 0.8603 | F1: 0.8603
# Val   Loss: 0.6120 | Acc: 0.8347 | F1: 0.8347

# Epoch 10/10
                                                           
# Train Loss: 0.4299 | Acc: 0.8778 | F1: 0.8778
# Val   Loss: 0.7399 | Acc: 0.8000 | F1: 0.8000

# Best Validation F1: 0.8347

# 21m

Device: cuda
Training on device: cuda


C:\Users\pong\AppData\Local\Temp\ipykernel_4152\1086285773.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())



Epoch 1/10


Train Loss: 3.2094 | Acc: 0.1162 | F1: 0.1162
Val   Loss: 2.9013 | Acc: 0.1997 | F1: 0.1997

Epoch 2/10


Train Loss: 2.5426 | Acc: 0.2999 | F1: 0.2999
Val   Loss: 2.2737 | Acc: 0.3802 | F1: 0.3802

Epoch 3/10


...

Train Loss: 0.7785 | Acc: 0.7822 | F1: 0.7822
Val   Loss: 0.7843 | Acc: 0.7883 | F1: 0.7883

Epoch 7/10


Train Loss: 0.6682 | Acc: 0.8120 | F1: 0.8120
Val   Loss: 0.6884 | Acc: 0.8128 | F1: 0.8128

Epoch 8/10


Train Loss: 0.5562 | Acc: 0.8413 | F1: 0.8413
Val   Loss: 0.6492 | Acc: 0.8238 | F1: 0.8238

Epoch 9/10


Train Loss: 0.4902 | Acc: 0.8603 | F1: 0.8603
Val   Loss: 0.6120 | Acc: 0.8347 | F1: 0.8347

Epoch 10/10


Train Loss: 0.4299 | Acc: 0.8778 | F1: 0.8778
Val   Loss: 0.7399 | Acc: 0.8000 | F1: 0.8000

Best Validation F1: 0.8347


Можно заметить, что как будто не хватило количесвто эпох, но даже если увеличить, не получится выбить скор 0.97+

In [92]:
train_dataset = SimpsonsDataset(train_files, label_encoder = label_encoder, mode='train', augment_type='custom') 
val_dataset = SimpsonsDataset(val_files, label_encoder, mode='val')
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
loaders = {'train':train_loader, 'val': val_loader}

augment_type='custom' - добавление агументации 

In [93]:

# Параметры

SEED = 42
num_epochs = 12
lr_head = 1e-3
lr_backbone = 1e-4
weight_decay = 1e-4
model_path = "best_resnet50.pth"

# Воспроизводимость и устройство
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else 
                      "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


# Модель с более сильным класификатором 

def build_resnet50(num_classes, pretrained=True, dropout_p=0.4):
    model = models.resnet50(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_p),
        nn.Linear(1024, num_classes)
    )
    return model

num_classes = len(label_encoder.classes_)
model = build_resnet50(num_classes=num_classes, pretrained=True)
model = model.to(device)

# Оптимизатор
backbone_params = [p for n,p in model.named_parameters() if not n.startswith("fc.")]
head_params = [p for n,p in model.named_parameters() if n.startswith("fc.")]

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": lr_backbone},
    {"params": head_params, "lr": lr_head}
], weight_decay=weight_decay)

criterion = nn.CrossEntropyLoss()


#  Заморозка backbone 
for name, param in model.named_parameters():
    if not name.startswith("fc."):
        param.requires_grad = False


# Запуск обучения 
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs,
    device=device
)

print("Training finished. Best model saved as 'best_model.pth'.")

# Epoch 1/12
                                                           
# Train Loss: 1.4530 | Acc: 0.6179 | F1: 0.6179
# Val   Loss: 1.1574 | Acc: 0.6846 | F1: 0.6846

# Epoch 2/12
                                                           
# Train Loss: 0.9814 | Acc: 0.7276 | F1: 0.7276
# Val   Loss: 1.1201 | Acc: 0.6830 | F1: 0.6830

# Epoch 3/12
                                                           
# Train Loss: 0.8524 | Acc: 0.7561 | F1: 0.7561
# Val   Loss: 0.9870 | Acc: 0.7214 | F1: 0.7214

# Epoch 4/12
                                                           
# Train Loss: 0.7793 | Acc: 0.7732 | F1: 0.7732
# Val   Loss: 0.9109 | Acc: 0.7447 | F1: 0.7447

# Epoch 5/12
                                                           
# Train Loss: 0.7246 | Acc: 0.7911 | F1: 0.7911
# Val   Loss: 0.8627 | Acc: 0.7514 | F1: 0.7514

# Epoch 6/12
                                                           
# Train Loss: 0.6726 | Acc: 0.8035 | F1: 0.8035
# Val   Loss: 0.8305 | Acc: 0.7644 | F1: 0.7644

# Epoch 7/12
                                                           
# Train Loss: 0.6238 | Acc: 0.8139 | F1: 0.8139
# Val   Loss: 0.8840 | Acc: 0.7461 | F1: 0.7461

# Epoch 8/12
                                                           
# Train Loss: 0.6001 | Acc: 0.8251 | F1: 0.8251
# Val   Loss: 0.8340 | Acc: 0.7639 | F1: 0.7639

# Epoch 9/12
                                                           
# Train Loss: 0.5575 | Acc: 0.8344 | F1: 0.8344
# Val   Loss: 0.8114 | Acc: 0.7696 | F1: 0.7696

# Epoch 10/12
                                                           
# Train Loss: 0.5256 | Acc: 0.8429 | F1: 0.8429
# Val   Loss: 0.7733 | Acc: 0.7826 | F1: 0.7826

# Epoch 11/12
                                                           
# Train Loss: 0.4983 | Acc: 0.8502 | F1: 0.8502
# Val   Loss: 0.7861 | Acc: 0.7782 | F1: 0.7782

# Epoch 12/12
                                                           
# Train Loss: 0.4869 | Acc: 0.8503 | F1: 0.8503
# Val   Loss: 0.7487 | Acc: 0.7872 | F1: 0.7872

# 40m


c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\pong/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:10<00:00, 9.83MB/s]


Training on device: cuda

Epoch 1/12


Train Loss: 1.4530 | Acc: 0.6179 | F1: 0.6179
Val   Loss: 1.1574 | Acc: 0.6846 | F1: 0.6846

Epoch 2/12


...

Train Loss: 0.6001 | Acc: 0.8251 | F1: 0.8251
Val   Loss: 0.8340 | Acc: 0.7639 | F1: 0.7639

Epoch 9/12


Train Loss: 0.5575 | Acc: 0.8344 | F1: 0.8344
Val   Loss: 0.8114 | Acc: 0.7696 | F1: 0.7696

Epoch 10/12


Train Loss: 0.5256 | Acc: 0.8429 | F1: 0.8429
Val   Loss: 0.7733 | Acc: 0.7826 | F1: 0.7826

Epoch 11/12


Train Loss: 0.4983 | Acc: 0.8502 | F1: 0.8502
Val   Loss: 0.7861 | Acc: 0.7782 | F1: 0.7782

Epoch 12/12


Train Loss: 0.4869 | Acc: 0.8503 | F1: 0.8503
Val   Loss: 0.7487 | Acc: 0.7872 | F1: 0.7872

Best Validation F1: 0.7872
Training finished. Best model saved as 'best_model.pth'.


Скор получился ещё хуже( 

In [94]:

num_classes = len(label_encoder.classes_)
model = build_resnet50(num_classes=num_classes, pretrained=True)
model = model.to(device)

backbone_params = [p for n,p in model.named_parameters() if not n.startswith("fc.")]
head_params = [p for n,p in model.named_parameters() if n.startswith("fc.")]
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": lr_backbone},
    {"params": head_params, "lr": lr_head}
], weight_decay=weight_decay)

criterion = nn.CrossEntropyLoss()

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs,
    device=device
)

torch.save(model.state_dict(), model_path)
print("Training finished. Best model saved as", model_path)

# Epoch 1/12
                                                           
# Train Loss: 0.5897 | Acc: 0.8612 | F1: 0.8612
# Val   Loss: 0.2056 | Acc: 0.9454 | F1: 0.9454

# Epoch 2/12
                                                           
# Train Loss: 0.1388 | Acc: 0.9654 | F1: 0.9654
# Val   Loss: 0.1698 | Acc: 0.9585 | F1: 0.9585

# Epoch 3/12
                                                           
# Train Loss: 0.0798 | Acc: 0.9796 | F1: 0.9796
# Val   Loss: 0.1365 | Acc: 0.9669 | F1: 0.9669

# Epoch 4/12
                                                           
# Train Loss: 0.0548 | Acc: 0.9844 | F1: 0.9844
# Val   Loss: 0.1504 | Acc: 0.9643 | F1: 0.9643

# Epoch 5/12
                                                           
# Train Loss: 0.0420 | Acc: 0.9890 | F1: 0.9890
# Val   Loss: 0.1267 | Acc: 0.9694 | F1: 0.9694

# Epoch 6/12
                                                           
# Train Loss: 0.0362 | Acc: 0.9894 | F1: 0.9894
# Val   Loss: 0.1193 | Acc: 0.9710 | F1: 0.9710

# Epoch 7/12
                                                           
# Train Loss: 0.0282 | Acc: 0.9920 | F1: 0.9920
# Val   Loss: 0.1217 | Acc: 0.9748 | F1: 0.9748

# Epoch 8/12
                                                           
# Train Loss: 0.0260 | Acc: 0.9922 | F1: 0.9922
# Val   Loss: 0.1333 | Acc: 0.9702 | F1: 0.9702

# Epoch 9/12
                                                           
# Train Loss: 0.0304 | Acc: 0.9912 | F1: 0.9912
# Val   Loss: 0.1249 | Acc: 0.9740 | F1: 0.9740

# Epoch 10/12
                                                           
# Train Loss: 0.0225 | Acc: 0.9935 | F1: 0.9935
# Val   Loss: 0.1515 | Acc: 0.9702 | F1: 0.9702

# Epoch 11/12
                                                           
# Train Loss: 0.0246 | Acc: 0.9929 | F1: 0.9929
# Val   Loss: 0.1675 | Acc: 0.9639 | F1: 0.9639

# Epoch 12/12
                                                           
# Train Loss: 0.0278 | Acc: 0.9917 | F1: 0.9917
# Val   Loss: 0.1791 | Acc: 0.9635 | F1: 0.9635

# 32m


c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Training on device: cuda

Epoch 1/12


Train Loss: 0.5897 | Acc: 0.8612 | F1: 0.8612
Val   Loss: 0.2056 | Acc: 0.9454 | F1: 0.9454

Epoch 2/12


Train Loss: 0.1388 | Acc: 0.9654 | F1: 0.9654
Val   Loss: 0.1698 | Acc: 0.9585 | F1: 0.9585

Epoch 3/12


Train Loss: 0.0798 | Acc: 0.9796 | F1: 0.9796
Val   Loss: 0.1365 | Acc: 0.9669 | F1: 0.9669

Epoch 4/12


...

Train Loss: 0.0260 | Acc: 0.9922 | F1: 0.9922
Val   Loss: 0.1333 | Acc: 0.9702 | F1: 0.9702

Epoch 9/12


Train Loss: 0.0304 | Acc: 0.9912 | F1: 0.9912
Val   Loss: 0.1249 | Acc: 0.9740 | F1: 0.9740

Epoch 10/12


Train Loss: 0.0225 | Acc: 0.9935 | F1: 0.9935
Val   Loss: 0.1515 | Acc: 0.9702 | F1: 0.9702

Epoch 11/12


Train Loss: 0.0246 | Acc: 0.9929 | F1: 0.9929
Val   Loss: 0.1675 | Acc: 0.9639 | F1: 0.9639

Epoch 12/12


Train Loss: 0.0278 | Acc: 0.9917 | F1: 0.9917
Val   Loss: 0.1791 | Acc: 0.9635 | F1: 0.9635

Best Validation F1: 0.9748
Training finished. Best model saved as best_resnet50.pth


In [95]:
# Ну вот, на конец-то хорошие результаты, продолжим эксперементы 
# Попробую всё тоже самое, только не используя агументацию 

In [96]:
train_dataset = SimpsonsDataset(train_files, label_encoder = label_encoder, mode='train', augment_type='custom')
val_dataset = SimpsonsDataset(val_files, label_encoder, mode='val')
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
loaders = {'train':train_loader, 'val': val_loader}

In [97]:
torch.cuda.empty_cache() #очищаем кэш

#Пересоздаём модель и оптимизатор
model = build_resnet50(num_classes=num_classes, pretrained=True)
model = model.to(device)

backbone_params = [p for n,p in model.named_parameters() if not n.startswith("fc.")]
head_params = [p for n,p in model.named_parameters() if n.startswith("fc.")]
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": lr_backbone},
    {"params": head_params, "lr": lr_head}
], weight_decay=weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=7,
    device=device
)

torch.save(model.state_dict(), model_path)
print("Training finished. Best model saved as", model_path)

# Epoch 1/7
                                                           
# Train Loss: 0.5964 | Acc: 0.8546 | F1: 0.8546
# Val   Loss: 0.2007 | Acc: 0.9478 | F1: 0.9478

# Epoch 2/7
                                                           
# Train Loss: 0.1387 | Acc: 0.9642 | F1: 0.9642
# Val   Loss: 0.1690 | Acc: 0.9568 | F1: 0.9568

# Epoch 3/7
                                                           
# Train Loss: 0.0765 | Acc: 0.9807 | F1: 0.9807
# Val   Loss: 0.1512 | Acc: 0.9610 | F1: 0.9610

# Epoch 4/7
                                                           
# Train Loss: 0.0553 | Acc: 0.9858 | F1: 0.9858
# Val   Loss: 0.1490 | Acc: 0.9654 | F1: 0.9654

# Epoch 5/7
                                                           
# Train Loss: 0.0396 | Acc: 0.9888 | F1: 0.9888
# Val   Loss: 0.1810 | Acc: 0.9583 | F1: 0.9583

# Epoch 6/7
                                                           
# Train Loss: 0.0376 | Acc: 0.9896 | F1: 0.9896
# Val   Loss: 0.1315 | Acc: 0.9687 | F1: 0.9687

# Epoch 7/7
                                                           
# Train Loss: 0.0315 | Acc: 0.9906 | F1: 0.9906
# Val   Loss: 0.1433 | Acc: 0.9683 | F1: 0.9683

# 17m

c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Training on device: cuda

Epoch 1/7


Train Loss: 0.5964 | Acc: 0.8546 | F1: 0.8546
Val   Loss: 0.2007 | Acc: 0.9478 | F1: 0.9478

Epoch 2/7


Train Loss: 0.1387 | Acc: 0.9642 | F1: 0.9642
Val   Loss: 0.1690 | Acc: 0.9568 | F1: 0.9568

Epoch 3/7


Train Loss: 0.0765 | Acc: 0.9807 | F1: 0.9807
Val   Loss: 0.1512 | Acc: 0.9610 | F1: 0.9610

Epoch 4/7


Train Loss: 0.0553 | Acc: 0.9858 | F1: 0.9858
Val   Loss: 0.1490 | Acc: 0.9654 | F1: 0.9654

Epoch 5/7


Train Loss: 0.0396 | Acc: 0.9888 | F1: 0.9888
Val   Loss: 0.1810 | Acc: 0.9583 | F1: 0.9583

Epoch 6/7


Train Loss: 0.0376 | Acc: 0.9896 | F1: 0.9896
Val   Loss: 0.1315 | Acc: 0.9687 | F1: 0.9687

Epoch 7/7


Train Loss: 0.0315 | Acc: 0.9906 | F1: 0.9906
Val   Loss: 0.1433 | Acc: 0.9683 | F1: 0.9683

Best Validation F1: 0.9687
Training finished. Best model saved as best_resnet50.pth


Аугментация всё таки приносила свои плоды, так что вернём её и попробуем метод mixup augmentation

Но для этого надо немного изменить функции обучения

In [100]:
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_one_epoch(model, dataloader, loss_func, optimizer, device, use_mixup=False, alpha=0.4):
    model.train()
    total_loss, total_correct, n = 0, 0, 0

    for images, labels in tqdm(dataloader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if use_mixup:
            mixed_x, y_a, y_b, lam = mixup_data(images, labels, alpha)
            outputs = model(mixed_x)
            loss = mixup_criterion(loss_func, outputs, y_a, y_b, lam)
        else:
            outputs = model(images)
            loss = loss_func(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        if not use_mixup:
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            n += labels.size(0)

    avg_loss = total_loss / len(dataloader.dataset)
    avg_acc = total_correct / n if n > 0 else 0.0
    return avg_loss, avg_acc

def evaluate_model(model, dataloader, loss_func, device):
    model.eval()
    total_loss, total_correct, n = 0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_func(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            n += labels.size(0)
    avg_loss = total_loss / len(dataloader.dataset)
    avg_acc = total_correct / n
    return avg_loss, avg_acc

def train_model(model, train_loader, val_loader, loss_func, optimizer, num_epochs, device, scheduler=None, use_mixup=False):
    best_val_acc = 0
    history = {"train_acc": [], "val_acc": []}

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_func, optimizer, device, use_mixup=use_mixup)
        val_loss, val_acc = evaluate_model(model, val_loader, loss_func, device)

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if scheduler:
            scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_mixup_model.pth")

        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f}\n")

    return model, history


In [101]:
torch.cuda.empty_cache() #очищаем кэш

#Пересоздаём модель и оптимизатор
model = build_resnet50(num_classes=num_classes, pretrained=True)
model = model.to(device)

backbone_params = [p for n,p in model.named_parameters() if not n.startswith("fc.")]
head_params = [p for n,p in model.named_parameters() if n.startswith("fc.")]
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": lr_backbone},
    {"params": head_params, "lr": lr_head}
], weight_decay=weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs,
    device=device,
    scheduler=scheduler,
    use_mixup=True   
)

torch.save(model.state_dict(), model_path)
print("Training finished. Best model saved as", model_path)

# Epoch 1/12
# Train Loss: 1.4056 | Acc: 0.0000
# Val   Loss: 0.3726 | Acc: 0.9337

                                                           
# Epoch 2/12
# Train Loss: 0.9968 | Acc: 0.0000
# Val   Loss: 0.2979 | Acc: 0.9499

                                                           
# Epoch 3/12
# Train Loss: 0.8545 | Acc: 0.0000
# Val   Loss: 0.2709 | Acc: 0.9568

                                                           
# Epoch 4/12
# Train Loss: 0.9085 | Acc: 0.0000
# Val   Loss: 0.2230 | Acc: 0.9704

                                                           
# Epoch 5/12
# Train Loss: 0.7859 | Acc: 0.0000
# Val   Loss: 0.1977 | Acc: 0.9738

                                                           
# Epoch 6/12
# Train Loss: 0.7829 | Acc: 0.0000
# Val   Loss: 0.1735 | Acc: 0.9773

                                                           
# Epoch 7/12
# Train Loss: 0.7032 | Acc: 0.0000
# Val   Loss: 0.1889 | Acc: 0.9778

                                                           
# Epoch 8/12
# Train Loss: 0.6543 | Acc: 0.0000
# Val   Loss: 0.1510 | Acc: 0.9799

                                                           
# Epoch 9/12
# Train Loss: 0.6128 | Acc: 0.0000
# Val   Loss: 0.1557 | Acc: 0.9797

                                                           
# Epoch 10/12
# Train Loss: 0.6787 | Acc: 0.0000
# Val   Loss: 0.1747 | Acc: 0.9792

                                                           
# Epoch 11/12
# Train Loss: 0.6393 | Acc: 0.0000
# Val   Loss: 0.1378 | Acc: 0.9832

                                                           
# Epoch 12/12
# Train Loss: 0.6541 | Acc: 0.0000
# Val   Loss: 0.1357 | Acc: 0.9826

# 25m

c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\pong\anaconda3\envs\ml\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/12
Train Loss: 1.4056 | Acc: 0.0000
Val   Loss: 0.3726 | Acc: 0.9337



Epoch 2/12
Train Loss: 0.9968 | Acc: 0.0000
Val   Loss: 0.2979 | Acc: 0.9499



Epoch 3/12
Train Loss: 0.8545 | Acc: 0.0000
Val   Loss: 0.2709 | Acc: 0.9568



Epoch 4/12
Train Loss: 0.9085 | Acc: 0.0000
Val   Loss: 0.2230 | Acc: 0.9704



...

Epoch 8/12
Train Loss: 0.6543 | Acc: 0.0000
Val   Loss: 0.1510 | Acc: 0.9799



Epoch 9/12
Train Loss: 0.6128 | Acc: 0.0000
Val   Loss: 0.1557 | Acc: 0.9797



Epoch 10/12
Train Loss: 0.6787 | Acc: 0.0000
Val   Loss: 0.1747 | Acc: 0.9792



Epoch 11/12
Train Loss: 0.6393 | Acc: 0.0000
Val   Loss: 0.1378 | Acc: 0.9832



Epoch 12/12
Train Loss: 0.6541 | Acc: 0.0000
Val   Loss: 0.1357 | Acc: 0.9826

Training finished. Best model saved as best_resnet50.pth


Точность на валидации немного выше, а вот на лидерборде такая же( - 0.99468


Теперь попробуем настоящую тяжёлую артилерию - ResNet152, если kaggle позволит)

Сразу не позвол, и после перезапуска( стерание всех возможных кешей и загрухок, тоже)  
Попытка 3 тоже не увенчалась успехом( Провал, ничего не помогло, поробуем ResNet101

Эх на ResNet101 тоже памяти не хватило

In [113]:
train_dataset = SimpsonsDataset(train_files, label_encoder = label_encoder, mode='train', augment_type='custom')
val_dataset = SimpsonsDataset(val_files, label_encoder, mode='val')
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
loaders = {'train':train_loader, 'val': val_loader}

In [ ]:
# Очищаем кэш
torch.cuda.empty_cache()

# Параметры (настройте под свою задачу)
# num_classes = len(label_encoder.classes_)
# lr_backbone = 1e-5  # ResNet-101 может требовать чуть меньший lr для backbone
# lr_head = 1e-4
# weight_decay = 1e-4
# num_epochs = 30

SEED = 42
num_epochs = 30
lr_head = 1e-4 # lr_head = 1e-4
lr_backbone = 1e-5 # lr_backbone = 1e-5 
weight_decay = 1e-4
num_classes = len(label_encoder.classes_)
model_path = "best_resnet101.pth"

# Создаем модель ResNet-101
def build_resnet101(num_classes=num_classes, pretrained=True):
    """
    Создает модель ResNet-101 с возможностью настройки количества классов
    """
    if pretrained:
        model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet101(weights=None)
    
    # Заменяем последний fully connected слой
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

In [121]:
# Создаем модель
model = build_resnet101(num_classes=num_classes, pretrained=True)
model = model.to(device)

# Разделяем параметры для разных learning rates
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
head_params = [p for n, p in model.named_parameters() if n.startswith("fc.")]

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": lr_backbone},
    {"params": head_params, "lr": lr_head}
], weight_decay=weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Функция потерь
criterion = nn.CrossEntropyLoss()

In [122]:
# Запуск обучения
model, history = train_model(
    model=model,
    train_loader=train_loader,  # Ваш train_loader
    val_loader=val_loader,      # Ваш val_loader
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs,
    device=device,
    scheduler=scheduler,
    use_mixup=True   # Включен Mixup для лучшей обобщаемости
)

# Epoch 1/30
# Train Loss: 2.2187 | Acc: 0.0000
# Val   Loss: 0.8119 | Acc: 0.8535

                                                           
# Epoch 2/30
# Train Loss: 1.2112 | Acc: 0.0000
# Val   Loss: 0.4351 | Acc: 0.9219

                                                           
# Epoch 3/30
# Train Loss: 1.0161 | Acc: 0.0000
# Val   Loss: 0.3000 | Acc: 0.9429

                                                           
# Epoch 4/30
# Train Loss: 1.0077 | Acc: 0.0000
# Val   Loss: 0.2819 | Acc: 0.9541

                                                           
# Epoch 5/30
# Train Loss: 0.8922 | Acc: 0.0000
# Val   Loss: 0.2541 | Acc: 0.9605

                                                           
# Epoch 6/30
# Train Loss: 0.8880 | Acc: 0.0000
# Val   Loss: 0.2442 | Acc: 0.9656

                                                           
# Epoch 7/30
# Train Loss: 0.8956 | Acc: 0.0000
# Val   Loss: 0.2448 | Acc: 0.9696

                                                           
# Epoch 8/30
# Train Loss: 0.8284 | Acc: 0.0000
# Val   Loss: 0.2125 | Acc: 0.9725

                                                           
# Epoch 9/30
# Train Loss: 0.8667 | Acc: 0.0000
# Val   Loss: 0.1680 | Acc: 0.9763

                                                           
# Epoch 10/30
# Train Loss: 0.7610 | Acc: 0.0000
# Val   Loss: 0.1644 | Acc: 0.9773

                                                           
# Epoch 11/30
# Train Loss: 0.7493 | Acc: 0.0000
# Val   Loss: 0.1515 | Acc: 0.9790

                                                           
# Epoch 12/30
# Train Loss: 0.7416 | Acc: 0.0000
# Val   Loss: 0.1639 | Acc: 0.9801

                                                           
# Epoch 13/30
# Train Loss: 0.8016 | Acc: 0.0000
# Val   Loss: 0.1526 | Acc: 0.9807

                                                           
# Epoch 14/30
# Train Loss: 0.7320 | Acc: 0.0000
# Val   Loss: 0.1722 | Acc: 0.9828

                                                           
# Epoch 15/30
# Train Loss: 0.7540 | Acc: 0.0000
# Val   Loss: 0.1610 | Acc: 0.9824

                                                           
# Epoch 16/30
# Train Loss: 0.7405 | Acc: 0.0000
# Val   Loss: 0.1745 | Acc: 0.9822

                                                           
# Epoch 17/30
# Train Loss: 0.7100 | Acc: 0.0000
# Val   Loss: 0.1341 | Acc: 0.9817

                                                           
# Epoch 18/30
# Train Loss: 0.6792 | Acc: 0.0000
# Val   Loss: 0.1301 | Acc: 0.9818

                                                           
# Epoch 19/30
# Train Loss: 0.7360 | Acc: 0.0000
# Val   Loss: 0.1627 | Acc: 0.9815

                                                           
# Epoch 20/30
# Train Loss: 0.7490 | Acc: 0.0000
# Val   Loss: 0.1641 | Acc: 0.9811

                                                           
# Epoch 21/30
# Train Loss: 0.7573 | Acc: 0.0000
# Val   Loss: 0.1502 | Acc: 0.9824

                                                           
# Epoch 22/30
# Train Loss: 0.7183 | Acc: 0.0000
# Val   Loss: 0.1422 | Acc: 0.9820

                                                           
# Epoch 23/30
# Train Loss: 0.7130 | Acc: 0.0000
# Val   Loss: 0.1403 | Acc: 0.9826

                                                           
# Epoch 24/30
# Train Loss: 0.6994 | Acc: 0.0000
# Val   Loss: 0.1345 | Acc: 0.9820

                                                           
# Epoch 25/30
# Train Loss: 0.7271 | Acc: 0.0000
# Val   Loss: 0.1459 | Acc: 0.9832

                                                           
# Epoch 26/30
# Train Loss: 0.7273 | Acc: 0.0000
# Val   Loss: 0.1477 | Acc: 0.9832

                                                           
# Epoch 27/30
# Train Loss: 0.6887 | Acc: 0.0000
# Val   Loss: 0.1328 | Acc: 0.9834

                                                           
# Epoch 28/30
# Train Loss: 0.6593 | Acc: 0.0000
# Val   Loss: 0.1365 | Acc: 0.9828

                                                           
# Epoch 29/30
# Train Loss: 0.6932 | Acc: 0.0000
# Val   Loss: 0.1416 | Acc: 0.9832

                                                           
# Epoch 30/30
# Train Loss: 0.7237 | Acc: 0.0000
# Val   Loss: 0.1591 | Acc: 0.9818

# 64m

Epoch 1/30
Train Loss: 2.2187 | Acc: 0.0000
Val   Loss: 0.8119 | Acc: 0.8535



Epoch 2/30
Train Loss: 1.2112 | Acc: 0.0000
Val   Loss: 0.4351 | Acc: 0.9219



Epoch 3/30
Train Loss: 1.0161 | Acc: 0.0000
Val   Loss: 0.3000 | Acc: 0.9429



Epoch 4/30
Train Loss: 1.0077 | Acc: 0.0000
Val   Loss: 0.2819 | Acc: 0.9541



Epoch 5/30
Train Loss: 0.8922 | Acc: 0.0000
Val   Loss: 0.2541 | Acc: 0.9605



...

Epoch 26/30
Train Loss: 0.7273 | Acc: 0.0000
Val   Loss: 0.1477 | Acc: 0.9832



Epoch 27/30
Train Loss: 0.6887 | Acc: 0.0000
Val   Loss: 0.1328 | Acc: 0.9834



Epoch 28/30
Train Loss: 0.6593 | Acc: 0.0000
Val   Loss: 0.1365 | Acc: 0.9828



Epoch 29/30
Train Loss: 0.6932 | Acc: 0.0000
Val   Loss: 0.1416 | Acc: 0.9832



Epoch 30/30
Train Loss: 0.7237 | Acc: 0.0000
Val   Loss: 0.1591 | Acc: 0.9818



In [123]:
# Сохраняем финальную модель
model_path = "resnet101_mixup_final.pth"
torch.save(model.state_dict(), model_path)
print("Training finished. Best model saved as", model_path)

Training finished. Best model saved as resnet101_mixup_final.pth


In [124]:
predicted_numeric_labels = predict(model, test_loader)

Test mode...


  0%|          | 0/16 [00:00<?, ?it/s]

и преобразуем их в текстовые метки:

In [125]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)

Загрузим пример файла для загрузки на Kaggle (проверьте путь, по которому у вас лежит файл sample_submission.csv и при необходимости скорректируйте путь в коде ниже):

In [128]:
import pandas as pd
sample_submission = pd.read_csv("content/sample_submission.csv")
sample_submission.head(10)

,Id,Expected
0,img0.jpg,bart_simpson
1,img1.jpg,bart_simpson
2,img2.jpg,bart_simpson
3,img3.jpg,bart_simpson
4,img4.jpg,bart_simpson
5,img5.jpg,bart_simpson
6,img6.jpg,bart_simpson
7,img7.jpg,bart_simpson
8,img8.jpg,bart_simpson
9,img9.jpg,bart_simpson


In [129]:
my_submission = pd.DataFrame({'Id': [path.name for path in test_files], 'Expected': predicted_text_labels})
my_submission.head(10)

,Id,Expected
0,img0.jpg,nelson_muntz
1,img1.jpg,bart_simpson
2,img10.jpg,ned_flanders
3,img100.jpg,chief_wiggum
4,img101.jpg,apu_nahasapeemapetilon
5,img102.jpg,kent_brockman
6,img103.jpg,edna_krabappel
7,img104.jpg,chief_wiggum
8,img105.jpg,lisa_simpson
9,img106.jpg,kent_brockman


In [130]:
my_submission.to_csv('content/submission_resnet101.csv', index=False)

На kaggle 57 Nikita Kucherov 0.99574

In [135]:
# Очищаем кэш
torch.cuda.empty_cache()

# Параметры (скорректированы для ResNet-152)
SEED = 42
num_epochs = 40
# ResNet-152 требует чуть меньший learning rate из-за большей глубины
lr_head = 5e-5      # Уменьшен с 1e-4
lr_backbone = 5e-6  # Уменьшен с 1e-5
weight_decay = 1e-4
num_classes = len(label_encoder.classes_)
model_path = "best_resnet152.pth"

In [136]:
# Создаем модель ResNet-152
def build_resnet152(num_classes=num_classes, pretrained=True):
    """
    Создает модель ResNet-152 с возможностью настройки количества классов
    """
    if pretrained:
        # Используем предобученные веса ImageNet
        model = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet152(weights=None)
    
    # Заменяем последний fully connected слой
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

In [137]:
# Создаем модель
model = build_resnet152(num_classes=num_classes, pretrained=True)
model = model.to(device)

# Разделяем параметры для разных learning rates
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
head_params = [p for n, p in model.named_parameters() if n.startswith("fc.")]

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": lr_backbone},
    {"params": head_params, "lr": lr_head}
], weight_decay=weight_decay)

# Cosine annealing scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Функция потерь
criterion = nn.CrossEntropyLoss()

In [138]:
# Запуск обучения с Mixup
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs,
    device=device,
    scheduler=scheduler,
    use_mixup=True   # Mixup помогает бороться с переобучением глубоких сетей
)

# Epoch 1/40
# Train Loss: 2.7020 | Acc: 0.0000
# Val   Loss: 1.5114 | Acc: 0.7489

                                                           
# Epoch 2/40
# Train Loss: 1.5611 | Acc: 0.0000
# Val   Loss: 0.7567 | Acc: 0.8594

                                                           
# Epoch 3/40
# Train Loss: 1.3406 | Acc: 0.0000
# Val   Loss: 0.5663 | Acc: 0.9003

                                                           
# Epoch 4/40
# Train Loss: 1.1601 | Acc: 0.0000
# Val   Loss: 0.4263 | Acc: 0.9280

                                                           
# Epoch 5/40
# Train Loss: 1.0320 | Acc: 0.0000
# Val   Loss: 0.3606 | Acc: 0.9383

                                                           
# Epoch 6/40
# Train Loss: 0.9668 | Acc: 0.0000
# Val   Loss: 0.3221 | Acc: 0.9469

                                                           
# Epoch 7/40
# Train Loss: 0.9603 | Acc: 0.0000
# Val   Loss: 0.2831 | Acc: 0.9543

                                                           
# Epoch 8/40
# Train Loss: 0.9093 | Acc: 0.0000
# Val   Loss: 0.2954 | Acc: 0.9578

                                                           
# Epoch 9/40
# Train Loss: 0.9062 | Acc: 0.0000
# Val   Loss: 0.2498 | Acc: 0.9656

                                                           
# Epoch 10/40
# Train Loss: 0.9065 | Acc: 0.0000
# Val   Loss: 0.2679 | Acc: 0.9650

                                                           
# Epoch 11/40
# Train Loss: 0.8561 | Acc: 0.0000
# Val   Loss: 0.2451 | Acc: 0.9689

                                                           
# Epoch 12/40
# Train Loss: 0.8020 | Acc: 0.0000
# Val   Loss: 0.2217 | Acc: 0.9733

                                                           
# Epoch 13/40
# Train Loss: 0.8098 | Acc: 0.0000
# Val   Loss: 0.2313 | Acc: 0.9727

                                                           
# Epoch 14/40
# Train Loss: 0.7173 | Acc: 0.0000
# Val   Loss: 0.1728 | Acc: 0.9754

                                                           
# Epoch 15/40
# Train Loss: 0.7884 | Acc: 0.0000
# Val   Loss: 0.1625 | Acc: 0.9771

                                                           
# Epoch 16/40
# Train Loss: 0.8461 | Acc: 0.0000
# Val   Loss: 0.1612 | Acc: 0.9761

                                                           
# Epoch 17/40
# Train Loss: 0.7228 | Acc: 0.0000
# Val   Loss: 0.1516 | Acc: 0.9763

                                                           
# Epoch 18/40
# Train Loss: 0.8033 | Acc: 0.0000
# Val   Loss: 0.1729 | Acc: 0.9769

                                                           
# Epoch 19/40
# Train Loss: 0.8031 | Acc: 0.0000
# Val   Loss: 0.2052 | Acc: 0.9769

                                                           
# Epoch 20/40
# Train Loss: 0.7919 | Acc: 0.0000
# Val   Loss: 0.1813 | Acc: 0.9778

                                                           
# Epoch 21/40
# Train Loss: 0.7468 | Acc: 0.0000
# Val   Loss: 0.1622 | Acc: 0.9775

                                                           
# Epoch 22/40
# Train Loss: 0.7674 | Acc: 0.0000
# Val   Loss: 0.1669 | Acc: 0.9782

                                                           
# Epoch 23/40
# Train Loss: 0.7267 | Acc: 0.0000
# Val   Loss: 0.1779 | Acc: 0.9786

                                                           
# Epoch 24/40
# Train Loss: 0.7715 | Acc: 0.0000
# Val   Loss: 0.1574 | Acc: 0.9794

                                                           
# Epoch 25/40
# Train Loss: 0.7059 | Acc: 0.0000
# Val   Loss: 0.1909 | Acc: 0.9788

                                                           
# Epoch 26/40
# Train Loss: 0.7673 | Acc: 0.0000
# Val   Loss: 0.1410 | Acc: 0.9792

                                                           
# Epoch 27/40
# Train Loss: 0.7408 | Acc: 0.0000
# Val   Loss: 0.1517 | Acc: 0.9805

                                                           
# Epoch 28/40
# Train Loss: 0.6596 | Acc: 0.0000
# Val   Loss: 0.1400 | Acc: 0.9811

                                                           
# Epoch 29/40
# Train Loss: 0.7241 | Acc: 0.0000
# Val   Loss: 0.1408 | Acc: 0.9811

                                                           
# Epoch 30/40
# Train Loss: 0.7389 | Acc: 0.0000
# Val   Loss: 0.1605 | Acc: 0.9815

                                                           
# Epoch 31/40
# Train Loss: 0.6994 | Acc: 0.0000
# Val   Loss: 0.1367 | Acc: 0.9813

                                                           
# Epoch 32/40
# Train Loss: 0.7385 | Acc: 0.0000
# Val   Loss: 0.1388 | Acc: 0.9807

                                                           
# Epoch 33/40
# Train Loss: 0.8167 | Acc: 0.0000
# Val   Loss: 0.1902 | Acc: 0.9792

                                                           
# Epoch 34/40
# Train Loss: 0.7013 | Acc: 0.0000
# Val   Loss: 0.1351 | Acc: 0.9807

                                                           
# Epoch 35/40
# Train Loss: 0.7247 | Acc: 0.0000
# Val   Loss: 0.1298 | Acc: 0.9820

                                                           
# Epoch 36/40
# Train Loss: 0.7129 | Acc: 0.0000
# Val   Loss: 0.1586 | Acc: 0.9807

                                                           
# Epoch 37/40
# Train Loss: 0.7086 | Acc: 0.0000
# Val   Loss: 0.1373 | Acc: 0.9807

                                                           
# Epoch 38/40
# Train Loss: 0.7351 | Acc: 0.0000
# Val   Loss: 0.1631 | Acc: 0.9794

                                                           
# Epoch 39/40
# Train Loss: 0.7312 | Acc: 0.0000
# Val   Loss: 0.1388 | Acc: 0.9803

                                                           
# Epoch 40/40
# Train Loss: 0.7176 | Acc: 0.0000
# Val   Loss: 0.1297 | Acc: 0.9813

# 132m

Epoch 1/40
Train Loss: 2.7020 | Acc: 0.0000
Val   Loss: 1.5114 | Acc: 0.7489



Epoch 2/40
Train Loss: 1.5611 | Acc: 0.0000
Val   Loss: 0.7567 | Acc: 0.8594



Epoch 3/40
Train Loss: 1.3406 | Acc: 0.0000
Val   Loss: 0.5663 | Acc: 0.9003



Epoch 4/40
Train Loss: 1.1601 | Acc: 0.0000
Val   Loss: 0.4263 | Acc: 0.9280



Epoch 5/40
Train Loss: 1.0320 | Acc: 0.0000
Val   Loss: 0.3606 | Acc: 0.9383



...

Epoch 36/40
Train Loss: 0.7129 | Acc: 0.0000
Val   Loss: 0.1586 | Acc: 0.9807



Epoch 37/40
Train Loss: 0.7086 | Acc: 0.0000
Val   Loss: 0.1373 | Acc: 0.9807



Epoch 38/40
Train Loss: 0.7351 | Acc: 0.0000
Val   Loss: 0.1631 | Acc: 0.9794



Epoch 39/40
Train Loss: 0.7312 | Acc: 0.0000
Val   Loss: 0.1388 | Acc: 0.9803



Epoch 40/40
Train Loss: 0.7176 | Acc: 0.0000
Val   Loss: 0.1297 | Acc: 0.9813



In [139]:
# Сохраняем модель
torch.save(model.state_dict(), model_path)
print(f"Training finished. Best model saved as {model_path}")

Training finished. Best model saved as best_resnet152.pth


In [140]:
predicted_numeric_labels = predict(model, test_loader)

Test mode...


  0%|          | 0/16 [00:00<?, ?it/s]

и преобразуем их в текстовые метки:

In [141]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)

Загрузим пример файла для загрузки на Kaggle (проверьте путь, по которому у вас лежит файл sample_submission.csv и при необходимости скорректируйте путь в коде ниже):

In [142]:
import pandas as pd
sample_submission = pd.read_csv("content/sample_submission.csv")
sample_submission.head(10)

,Id,Expected
0,img0.jpg,bart_simpson
1,img1.jpg,bart_simpson
2,img2.jpg,bart_simpson
3,img3.jpg,bart_simpson
4,img4.jpg,bart_simpson
5,img5.jpg,bart_simpson
6,img6.jpg,bart_simpson
7,img7.jpg,bart_simpson
8,img8.jpg,bart_simpson
9,img9.jpg,bart_simpson


In [143]:
my_submission = pd.DataFrame({'Id': [path.name for path in test_files], 'Expected': predicted_text_labels})
my_submission.head(10)

,Id,Expected
0,img0.jpg,nelson_muntz
1,img1.jpg,bart_simpson
2,img10.jpg,ned_flanders
3,img100.jpg,chief_wiggum
4,img101.jpg,apu_nahasapeemapetilon
5,img102.jpg,kent_brockman
6,img103.jpg,edna_krabappel
7,img104.jpg,chief_wiggum
8,img105.jpg,lisa_simpson
9,img106.jpg,kent_brockman


In [144]:
my_submission.to_csv('content/submission_resnet152.csv', index=False)

In [ ]:
# # Сценарий: вы обучили модель 30 эпох, хотите добавить еще 10 (additional_epochs = 10)

# # Загружаем сохраненную модель
# model = build_resnet152(num_classes=num_classes, pretrained=True)
# model.load_state_dict(torch.load("best_resnet152.pth"))
# model = model.to(device)

# # Продолжаем обучение
# additional_epochs = 10
# model, history_extended = train_model(
#     model=model,  # Загруженная модель
#     train_loader=train_loader,
#     val_loader=val_loader,
#     loss_func=criterion,
#     optimizer=optimizer,  # Можно использовать тот же или новый
#     num_epochs=additional_epochs,
#     device=device,
#     scheduler=scheduler,
#     use_mixup=True
# )

In [ ]:
# Дообучить на 10 эпохах

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=criterion,
    optimizer=optimizer,
    num_epochs=10,
    device=device,
    scheduler=scheduler,
    use_mixup=True   # Mixup помогает бороться с переобучением глубоких сетей
)

                                                           
# Epoch 1/10
# Train Loss: 0.6346 | Acc: 0.0000
# Val   Loss: 0.1581 | Acc: 0.9799

                                                           
# Epoch 2/10
# Train Loss: 0.6746 | Acc: 0.0000
# Val   Loss: 0.1494 | Acc: 0.9805

                                                           
# Epoch 3/10
# Train Loss: 0.6831 | Acc: 0.0000
# Val   Loss: 0.1548 | Acc: 0.9809

                                                           
# Epoch 4/10
# Train Loss: 0.7868 | Acc: 0.0000
# Val   Loss: 0.1452 | Acc: 0.9813

                                                           
# Epoch 5/10
# Train Loss: 0.7108 | Acc: 0.0000
# Val   Loss: 0.1736 | Acc: 0.9797

                                                           
# Epoch 6/10
# Train Loss: 0.6877 | Acc: 0.0000
# Val   Loss: 0.1403 | Acc: 0.9809

                                                           
# Epoch 7/10
# Train Loss: 0.7823 | Acc: 0.0000
# Val   Loss: 0.1607 | Acc: 0.9797

                                                           
# Epoch 8/10
# Train Loss: 0.6779 | Acc: 0.0000
# Val   Loss: 0.1605 | Acc: 0.9794

                                                           
# Epoch 9/10
# Train Loss: 0.7418 | Acc: 0.0000
# Val   Loss: 0.1529 | Acc: 0.9805

                                                           
# Epoch 10/10
# Train Loss: 0.7274 | Acc: 0.0000
# Val   Loss: 0.1639 | Acc: 0.9805

# Смысла нет дообучать

Epoch 1/10
Train Loss: 0.6346 | Acc: 0.0000
Val   Loss: 0.1581 | Acc: 0.9799



Epoch 2/10
Train Loss: 0.6746 | Acc: 0.0000
Val   Loss: 0.1494 | Acc: 0.9805



Epoch 3/10
Train Loss: 0.6831 | Acc: 0.0000
Val   Loss: 0.1548 | Acc: 0.9809



Epoch 4/10
Train Loss: 0.7868 | Acc: 0.0000
Val   Loss: 0.1452 | Acc: 0.9813



Epoch 5/10
Train Loss: 0.7108 | Acc: 0.0000
Val   Loss: 0.1736 | Acc: 0.9797



Epoch 6/10
Train Loss: 0.6877 | Acc: 0.0000
Val   Loss: 0.1403 | Acc: 0.9809



Epoch 7/10
Train Loss: 0.7823 | Acc: 0.0000
Val   Loss: 0.1607 | Acc: 0.9797



Epoch 8/10
Train Loss: 0.6779 | Acc: 0.0000
Val   Loss: 0.1605 | Acc: 0.9794



Epoch 9/10
Train Loss: 0.7418 | Acc: 0.0000
Val   Loss: 0.1529 | Acc: 0.9805



Epoch 10/10
Train Loss: 0.7274 | Acc: 0.0000
Val   Loss: 0.1639 | Acc: 0.9805

